In [2]:
import numpy as np

class AnnulusPart:

    def __init__(self, name, data, span="half"):
        self.name = name
        self.data = data
        self.span = span
        self.get_geometry()

    def get_geometry(self):
        center1 = self.data.get("center1", [0,0])
        center2 = self.data.get("center2", [0,0])
        Ro1 = self.data.get("outer_radius")
        Ro2 = self.data.get("outer_radius")
        Ri1 = Ro1 - self.data.get("thickness")
        Ri2 = Ro2 - self.data.get("thickness")
        self.geometry = {
            "center1": center1,
            "center2": center2,
            "Ro1": Ro1,
            "Ro2": Ro2,
            "Ri1": Ri1,
            "Ri2": Ri2
        }

    def create_part(self, modelName):
        m = mdb.models[modelName]
        sketch_name = '__profile__' + self
        s = m.ConstrainedSketch(name=sketch_name, sheetSize=200.0)
        s.setPrimaryObject(option=STANDALONE)

        g1 = s.ArcByCenterEnds(center=self.geometry["center1"], Point1 = (self.geometry["Ro1"], 0), Point2 = (0.0, self.geometry["Ro2"]), direction=COUNTERCLOCKWISE)
        g2 = s.ArcByCenterEnds(center=self.geometry["center2"], Point1 = (self.geometry["Ri1"], 0), Point2 = (0.0, self.geometry["Ri2"]), direction=COUNTERCLOCKWISE)

        if self.span == "half":
            g3 = s.Line(point1=(-self.geometry["Ro1"], 0.0), point2=(self.geometry["Ro1"], 0.0))
            s.autoTrimCurve(curve1=g1, point1=(self.geometry["center1"][0], -self.geometry["Ro2"]))
            s.autoTrimCurve(curve1=g2, point1=(self.geometry["center2"][0], -self.geometry["Ri2"]))
            s.autoTrimCurve(curve1=g3, point1=self.geometry["center1"])
        elif self.span == "quarter":
            g3 = s.Line(point1=(self.geometry["center1"][0], 0.0), point2=(self.geometry["Ro1"], 0.0))
            g4 = s.Line(point1=(self.geometry["center1"][0], 0.0), point2=(0.0, self.geometry["Ro2"]))
            s.autoTrimCurve(curve1=g1, point1=(self.geometry["center1"][0], -self.geometry["Ro2"]))
            s.autoTrimCurve(curve1=g2, point1=(self.geometry["center2"][0], -self.geometry["Ri2"]))
            s.autoTrimCurve(curve1=g3, point1=self.geometry["center1"])
            s.autoTrimCurve(curve1=g4, point1=self.geometry["center1"])
        p = m.Part(name=self.name, dimensionality=TWO_D_PLANAR, type=DEFORMABLE_BODY)
        p.BaseShell(sketch=s)
        s.unsetPrimaryObject()
        return p
 